In [1]:
from pathlib import Path
import pandas as pd

DATA = Path.home() / "Desktop/mimic-iv-clinical-database-demo-2/hosp"
DATA.exists()

True

## Get IV treatment


In [2]:
emar = pd.read_csv(
    DATA / "emar.csv",
    usecols=["hadm_id", "pharmacy_id", "charttime", "medication", "event_txt"],
    parse_dates=["charttime"],
)
rx = pd.read_csv(DATA / "prescriptions.csv", usecols=["pharmacy_id", "route"])
print(len(emar), len(rx))

35835 18087


In [ ]:
# merge all meds record for this admission  
one = emar[emar["hadm_id"] == 20297618].sort_values("charttime")   
len(one)

307

In [5]:
# merge with prescriptions to get route
print(rx["pharmacy_id"].duplicated().sum())           # check no duplicate
rx = rx.drop_duplicates("pharmacy_id")
one = one.merge(rx, on="pharmacy_id", how="left")
len(one)

2862


307

In [ ]:
# filter for IV treatment for hyperkalemia
name = one["medication"].str.lower().fillna("")
is_drug = name.str.contains("calcium gluconate|calcium chloride|insulin|dextrose 50")   # All IV treatment for hyperkalemia

is_iv = one["route"].fillna("").str.startswith("IV")   # route start with IV（IV、IV DRIP…）
GIVEN_EVENTS = ["Administered", "Administered in Other Location", "Partial Administered", "Delayed Administered"] 
is_given = one["event_txt"].isin(GIVEN_EVENTS)   # actually given

print(is_drug.sum(), is_iv.sum(), is_given.sum())
tx = one[is_drug & is_iv & is_given]
tx

18 117 189


,hadm_id,pharmacy_id,charttime,medication,event_txt,route
135,20297618.0,52861272.0,2180-02-21 17:48:00,Insulin,Administered,IV
136,20297618.0,40614649.0,2180-02-21 17:48:00,Dextrose 50%,Administered,IV
148,20297618.0,79008047.0,2180-02-21 22:50:00,Calcium Gluconate,Administered,IV


## Export JSON

In [8]:
tx[["charttime", "medication", "route"]].to_json(
    "data/tx_20297618.json", orient="records", date_format="iso"
)

## Add all other patients


In [9]:
allrx = emar.merge(rx, on="pharmacy_id", how="left")  
len(allrx)

35835

In [10]:
name = allrx["medication"].str.lower().fillna("")
is_drug = name.str.contains("calcium gluconate|calcium chloride|insulin|dextrose 50")   # All IV treatment for hyperkalemia
is_iv = allrx["route"].fillna("").str.startswith("IV")   # route start with IV（IV、IV DRIP…）
GIVEN_EVENTS = ["Administered", "Administered in Other Location", "Partial Administered", "Delayed Administered"]
is_given = allrx["event_txt"].isin(GIVEN_EVENTS)   # actually given
tx_all = allrx[is_drug & is_iv & is_given]
len(tx_all)

108

In [11]:
tx_all["medication"].value_counts()

medication
Calcium Gluconate sliding scale (Critical Care-Ionized calcium)    40
Dextrose 50%                                                       35
Calcium Gluconate                                                  21
Insulin (Regular) for Hyperkalemia                                  6
Insulin                                                             5
Calcium Gluconate Replacement (Oncology)                            1
Name: count, dtype: int64

In [13]:
def classify_drug(medication):
    name = medication.lower()                         
    if "calcium gluconate" in name or "calcium chloride" in name:
        return "Calcium"
    elif "insulin" in name:
        return "Insulin"
    elif "dextrose 50" in name:
        return "Dextrose 50%"
    else:
        return "Other"

tx_all = tx_all.copy()
tx_all["drug_class"] = tx_all["medication"].map(classify_drug)
tx_all["drug_class"].value_counts()

drug_class
Calcium         62
Dextrose 50%    35
Insulin         11
Name: count, dtype: int64

In [14]:
tx_all[["hadm_id", "charttime", "drug_class", "medication"]].to_json(
    "data/tx_all.json", orient="records", date_format="iso"
)